In [ ]:
# Imports and Config
import numpy as np
import joblib, os, warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, classification_report

proc_dir = "../data/processed"
model_dir  = "../models"
os.makedirs(model_dir, exist_ok=True)

class_names = ["Low", "Moderate", "High"]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

: 

In [ ]:
# Load Data
X_train = np.load(f"{proc_dir}/X_train.npy")
X_test  = np.load(f"{proc_dir}/X_test.npy")
y_train = np.load(f"{proc_dir}/y_train.npy")
y_test  = np.load(f"{proc_dir}/y_test.npy")
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

In [ ]:
# Hyperparameter Tuning - Random Forest
rf_param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [10, 15, 20, 25, None],
    "min_samples_split" : [2, 5, 10],
    "min_samples_leaf" : [1, 2, 4],
    "max_features" : ["sqrt", "log2", 0.5],
    "class_weight" : ["balanced", "balanced_subsample"],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring="f1_macro",
    cv=skf,
    verbose=2,
    random_state=42,
    n_jobs=-1
)
rf_search.fit(X_train, y_train)

print(f"\nBest RF params : {rf_search.best_params_}")
print(f"Best CV F1 : {rf_search.best_score_:.4f}")

In [ ]:
# Evaluasi RF Tuned pada Test Set
best_rf   = rf_search.best_estimator_
y_pred_rf = best_rf.predict(X_test)

print(f"Test F1-macro : {f1_score(y_test, y_pred_rf, average='macro'):.4f}")
print()
print(classification_report(y_test, y_pred_rf, target_names=class_names))

In [ ]:
# Hyperparameter Tuning - LightGBM
print("Tuning LightGBM (estimasi: 5–10 menit)...")

lgb_param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [5, 8, 10, 15, -1],
    "learning_rate" : [0.01, 0.05, 0.1, 0.2],
    "num_leaves" : [31, 63, 127, 255],
    "min_child_samples" : [10, 20, 50],
    "subsample" : [0.6, 0.8, 1.0],
    "colsample_bytree" : [0.6, 0.8, 1.0],
    "reg_alpha" : [0, 0.1, 0.5],
    "reg_lambda" : [0, 0.1, 0.5],
}

lgb_search = RandomizedSearchCV(
    LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1),
    param_distributions=lgb_param_dist,
    n_iter=30,
    scoring="f1_macro",
    cv=skf,
    verbose=2,
    random_state=42,
    n_jobs=-1
)
lgb_search.fit(X_train, y_train)

print(f"\nBest LGB params : {lgb_search.best_params_}")
print(f"Best CV F1 : {lgb_search.best_score_:.4f}")

In [ ]:
# Evaluasi LGB Tuned pada Test Set
best_lgb = lgb_search.best_estimator_
y_pred_lgb  = best_lgb.predict(X_test)

print(f"Test F1-macro : {f1_score(y_test, y_pred_lgb, average='macro'):.4f}")
print()
print(classification_report(y_test, y_pred_lgb, target_names=class_names))

In [ ]:
# Perbandingan Sebelum vs Sesudah Tuning
import pandas as pd

comparison = pd.DataFrame({
    "RF (default)" : {"F1-macro": f1_score(y_test,
                        RandomForestClassifier(n_estimators=200, class_weight="balanced",
                        random_state=42, n_jobs=-1).fit(X_train, y_train).predict(X_test),
                        average="macro")},
    "RF (tuned)" : {"F1-macro": f1_score(y_test, y_pred_rf,  average="macro")},
    "LGB (tuned)" : {"F1-macro": f1_score(y_test, y_pred_lgb, average="macro")},
}).T

print("Perbandingan F1-macro:")
display(comparison.round(4))

In [ ]:
# Pilih Best Model & Export Model
f1_rf  = f1_score(y_test, y_pred_rf,  average="macro")
f1_lgb = f1_score(y_test, y_pred_lgb, average="macro")

if f1_rf >= f1_lgb:
    best_model = best_rf
    best_model_name = "Random Forest (tuned)"
else:
    best_model = best_lgb
    best_model_name = "LightGBM (tuned)"

print(f"Best model : {best_model_name}")
print(f"F1-macro : {max(f1_rf, f1_lgb):.4f}")

# Export semua model & artifacts
joblib.dump(best_rf, f"{model_dir}/rf_tuned.pkl")
joblib.dump(best_lgb, f"{model_dir}/lgb_tuned.pkl")
joblib.dump(best_model, f"{model_dir}/best_model.pkl")

print("\nModel tersimpan:")
for f in os.listdir(model_dir):
    size = os.path.getsize(f"{model_dir}/{f}") / 1e6
    print(f"{f} ({size:.1f} MB)")

In [ ]:
# Verifikasi Model
# Pastikan model bisa di-load dan predict dengan benar
loaded_model = joblib.load(f"{model_dir}/best_model.pkl")
loaded_scaler = joblib.load(f"{model_dir}/scaler.pkl")
loaded_cols = joblib.load(f"{model_dir}/feature_cols.pkl")
loaded_target = joblib.load(f"{model_dir}/target_mapping.pkl")

# Test predict 5 sampel
sample_pred = loaded_model.predict(X_test[:5])
label_map   = {v: k for k, v in loaded_target.items()}
print("Sample predictions:")
for i, pred in enumerate(sample_pred):
    print(f"Sample {i+1}: {label_map[pred]} (actual: {label_map[y_test[i]]})")

print(f"\nModel verified! Siap dipakai di Streamlit.")
print(f"Features : {loaded_cols}")